# Samuel Oswaldo Avellaneda Guevara
# Fredy Mateo Niño Vasquez

In [ ]:
!pip -q install pyspark pandas requests
import pandas as pd, requests
BASE = "https://www.datos.gov.co/resource/kgxf-xxbe.csv"
muestra = pd.read_csv(BASE + "?$limit=5")
print(muestra.columns.tolist())
muestra.head()

['periodo', 'estu_tipodocumento', 'estu_consecutivo', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_cod_dane_establecimiento', 'cole_cod_dane_sede', 'cole_cod_depto_ubicacion', 'cole_cod_mcpio_ubicacion', 'cole_codigo_icfes', 'cole_depto_ubicacion', 'cole_genero', 'cole_jornada', 'cole_mcpio_ubicacion', 'cole_naturaleza', 'cole_nombre_establecimiento', 'cole_nombre_sede', 'cole_sede_principal', 'estu_cod_depto_presentacion', 'estu_cod_mcpio_presentacion', 'estu_cod_reside_depto', 'estu_cod_reside_mcpio', 'estu_depto_presentacion', 'estu_depto_reside', 'estu_estadoinvestigacion', 'estu_estudiante', 'estu_fechanacimiento', 'estu_genero', 'estu_mcpio_presentacion', 'estu_mcpio_reside', 'estu_nacionalidad', 'estu_pais_reside', 'estu_privado_libertad', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fami_tienelavadora', 'd

,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,...,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20131,CR,SB11201310000414,URBANO,N,B,ACADÉMICO,311848000812,311848000812,11,...,Si,Si,Si,B+,94,88,NaN,NaN,NaN,NaN
1,20194,TI,SB11201940464873,RURAL,N,A,TÉCNICO/ACADÉMICO,141016000305,241016000342,41,...,Si,Si,Si,B1,71,66,70.0,65.0,69.0,339.0
2,20194,TI,SB11201940464873,RURAL,N,A,TÉCNICO/ACADÉMICO,141016000305,241016000342,41,...,Si,Si,Si,B1,71,66,70.0,65.0,69.0,339.0
3,20122,TI,SB11201220204399,URBANO,N,A,TÉCNICO/ACADÉMICO,163130000718,163130000718,63,...,Si,Si,No,A1,48,45,NaN,NaN,NaN,NaN
4,20132,TI,SB11201320464198,URBANO,N,A,TÉCNICO,119001000559,119001000559,19,...,Si,Si,Si,A-,43,52,NaN,NaN,NaN,NaN


In [ ]:
cols = muestra.columns.tolist()
for c in cols:
    print(c)

col_depto = [c for c in cols if "depto" in c and "cole" in c]
print("\nColumna(s) de departamento del colegio:", col_depto)

r = requests.get(BASE, params={"$select": col_depto[0], "$group": col_depto[0], "$limit": 100})
print(r.text)

periodo
estu_tipodocumento
estu_consecutivo
cole_area_ubicacion
cole_bilingue
cole_calendario
cole_caracter
cole_cod_dane_establecimiento
cole_cod_dane_sede
cole_cod_depto_ubicacion
cole_cod_mcpio_ubicacion
cole_codigo_icfes
cole_depto_ubicacion
cole_genero
cole_jornada
cole_mcpio_ubicacion
cole_naturaleza
cole_nombre_establecimiento
cole_nombre_sede
cole_sede_principal
estu_cod_depto_presentacion
estu_cod_mcpio_presentacion
estu_cod_reside_depto
estu_cod_reside_mcpio
estu_depto_presentacion
estu_depto_reside
estu_estadoinvestigacion
estu_estudiante
estu_fechanacimiento
estu_genero
estu_mcpio_presentacion
estu_mcpio_reside
estu_nacionalidad
estu_pais_reside
estu_privado_libertad
fami_cuartoshogar
fami_educacionmadre
fami_educacionpadre
fami_estratovivienda
fami_personashogar
fami_tieneautomovil
fami_tienecomputador
fami_tieneinternet
fami_tienelavadora
desemp_ingles
punt_ingles
punt_matematicas
punt_sociales_ciudadanas
punt_c_naturales
punt_lectura_critica
punt_global

Columna(s) de de

In [ ]:
import os, io
os.makedirs("data/raw", exist_ok=True)

FILTRO = "cole_cod_depto_ubicacion='15'"

# 1) Cuántas filas hay de Boyacá
r = requests.get(BASE, params={"$select": "count(*)", "$where": FILTRO})
print("Filas de Boyacá en la fuente:", r.text)

# 2) Descargar por páginas de 50.000
partes, offset = [], 0
while True:
    r = requests.get(BASE, params={"$limit": 50000, "$offset": offset,
                                   "$order": ":id", "$where": FILTRO}, timeout=300)
    df = pd.read_csv(io.StringIO(r.text), dtype=str)
    if df.empty:
        break
    partes.append(df)
    offset += 50000
    print("Descargadas:", offset)

raw = pd.concat(partes, ignore_index=True)
raw.to_csv("data/raw/saber11_boyaca.csv", index=False)
print("Listo:", raw.shape)

Filas de Boyacá en la fuente: "count"
"215234"

Descargadas: 50000
Descargadas: 100000
Descargadas: 150000
Descargadas: 200000
Descargadas: 250000
Listo: (215234, 51)


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("Saber11_Boyaca").getOrCreate()
sdf = spark.read.csv("data/raw/saber11_boyaca.csv", header=True, inferSchema=True)

print("Filas:", sdf.count(), "| Columnas:", len(sdf.columns))
sdf.printSchema()

Filas: 215234 | Columnas: 51
root
 |-- periodo: integer (nullable = true)
 |-- estu_tipodocumento: string (nullable = true)
 |-- estu_consecutivo: string (nullable = true)
 |-- cole_area_ubicacion: string (nullable = true)
 |-- cole_bilingue: string (nullable = true)
 |-- cole_calendario: string (nullable = true)
 |-- cole_caracter: string (nullable = true)
 |-- cole_cod_dane_establecimiento: long (nullable = true)
 |-- cole_cod_dane_sede: long (nullable = true)
 |-- cole_cod_depto_ubicacion: integer (nullable = true)
 |-- cole_cod_mcpio_ubicacion: integer (nullable = true)
 |-- cole_codigo_icfes: integer (nullable = true)
 |-- cole_depto_ubicacion: string (nullable = true)
 |-- cole_genero: string (nullable = true)
 |-- cole_jornada: string (nullable = true)
 |-- cole_mcpio_ubicacion: string (nullable = true)
 |-- cole_naturaleza: string (nullable = true)
 |-- cole_nombre_establecimiento: string (nullable = true)
 |-- cole_nombre_sede: string (nullable = true)
 |-- cole_sede_principal

In [ ]:
n = sdf.count()

# 1) Nulos y tipo por columna
nulos = sdf.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns]).toPandas().T
nulos.columns = ["nulos"]
nulos["%_nulos"] = (nulos["nulos"] / n * 100).round(2)
nulos["tipo"] = [t for _, t in sdf.dtypes]
diag = nulos.sort_values("%_nulos", ascending=False)
display(diag)

# 2) Duplicados
dup_exactos = n - sdf.dropDuplicates().count()
dup_por_id = n - sdf.dropDuplicates(["estu_consecutivo"]).count()
print("Total filas:", n)
print("Duplicados exactos (fila completa):", dup_exactos)
print("Duplicados por estu_consecutivo:", dup_por_id)

# 3) Rango de los puntajes (para ver atípicos)
sdf.select("punt_global", "punt_matematicas", "punt_lectura_critica",
           "punt_ingles", "punt_sociales_ciudadanas", "punt_c_naturales").describe().show()

,nulos,%_nulos,tipo
punt_c_naturales,83150,38.63,int
punt_lectura_critica,83150,38.63,int
punt_global,83150,38.63,int
punt_sociales_ciudadanas,83150,38.63,int
cole_bilingue,17277,8.03,string
fami_educacionpadre,7459,3.47,string
fami_educacionmadre,7424,3.45,string
fami_estratovivienda,5522,2.57,string
fami_tienecomputador,5515,2.56,string
fami_tieneinternet,5067,2.35,string


Total filas: 215234
Duplicados exactos (fila completa): 38576
Duplicados por estu_consecutivo: 40415
+-------+-----------------+------------------+--------------------+-----------------+------------------------+------------------+
|summary|      punt_global|  punt_matematicas|punt_lectura_critica|      punt_ingles|punt_sociales_ciudadanas|  punt_c_naturales|
+-------+-----------------+------------------+--------------------+-----------------+------------------------+------------------+
|  count|           132084|            215234|              132084|           215231|                  132084|            132084|
|   mean|264.7891039035765| 51.97416486242654|  53.735388086369284|49.22221915058709|      51.333053208564245|52.838125738166625|
| stddev|46.21448101164247|11.603459844536257|   9.751057693033184| 10.7858859955691|       11.09577417899238| 9.630691036516541|
|    min|                0|               0.0|                   0|             -1.0|                       0|         

In [ ]:
# A) Nulos y escala de puntaje por periodo
sdf.groupBy("periodo").agg(
    F.count("*").alias("filas"),
    F.sum(F.col("punt_global").isNull().cast("int")).alias("global_nulo"),
    F.round(F.avg("punt_matematicas"), 1).alias("prom_mat"),
    F.max("punt_matematicas").alias("max_mat")
).orderBy("periodo").show(60)

# B) Valores imposibles
print("punt_ingles < 0:", sdf.filter(F.col("punt_ingles") < 0).count())
print("punt_global = 0:", sdf.filter(F.col("punt_global") == 0).count())

+-------+-----+-----------+--------+-------+
|periodo|filas|global_nulo|prom_mat|max_mat|
+-------+-----+-----------+--------+-------+
|  20101|  658|        658|    43.3|  68.03|
|  20102|20202|      20202|    51.3| 114.88|
|  20111|  487|        487|    45.3|  81.33|
|  20112|20235|      20235|    46.9|  102.0|
|  20121|  460|        460|    44.9|   75.0|
|  20122|22352|      22352|    47.7|  109.0|
|  20131|  340|        340|    44.1|   78.0|
|  20132|18109|      18109|    47.2|  100.0|
|  20141|  307|        307|    43.4|   73.0|
|  20142|17325|          0|    52.2|  100.0|
|  20151|  299|          0|    47.9|   79.0|
|  20152|16538|          0|    53.4|  100.0|
|  20162|16840|          0|    54.5|  100.0|
|  20172|16872|          0|    54.0|  100.0|
|  20181|   87|          0|    46.4|   72.0|
|  20194|33474|          0|    55.1|  100.0|
|  20201|   94|          0|    51.6|   71.0|
|  20211|   65|          0|    48.3|   73.0|
|  20221|  120|          0|    49.2|   68.0|
|  20224|3

In [ ]:
from pyspark.sql import functions as F

# Diagnóstico antes de modificar la limpieza

print("1. Total de registros originales:")
print(sdf.count())

# Filtrar los periodos desde 2014-2
df_periodos = sdf.filter(F.col("periodo") >= 20142)

print("\n2. Registros desde 2014-2:")
print(df_periodos.count())

# Identificar duplicados exactos
df_sin_duplicados = df_periodos.dropDuplicates()

print("\n3. Registros después de quitar duplicados exactos:")
print(df_sin_duplicados.count())

# Comprobar identificadores repetidos
duplicados_id = (
    df_sin_duplicados
    .groupBy("estu_consecutivo")
    .count()
    .filter(F.col("count") > 1)
)

print("\n4. Identificadores que aparecen más de una vez:")
print(duplicados_id.count())

# Mostrar algunos ejemplos
duplicados_id.show(10, truncate=False)

1. Total de registros originales:
215234

2. Registros desde 2014-2:
132084

3. Registros después de quitar duplicados exactos:
100049

4. Identificadores que aparecen más de una vez:
0
+----------------+-----+
|estu_consecutivo|count|
+----------------+-----+
+----------------+-----+



In [ ]:
from pyspark.sql import functions as F

# Crear columna de año
df_anios = df_sin_duplicados.withColumn(
    "anio",
    F.floor(F.col("periodo") / 10).cast("int")
)

# Contar estudiantes por año
tabla_anios = (
    df_anios
    .groupBy("anio")
    .agg(
        F.count("*").alias("total_estudiantes"),
        F.round(F.avg("punt_global"), 2).alias("promedio_global")
    )
    .orderBy("anio")
)

tabla_anios.show(30, truncate=False)

# Verificar total
print(
    "Total de registros:",
    df_anios.count()
)

+----+-----------------+---------------+
|anio|total_estudiantes|promedio_global|
+----+-----------------+---------------+
|2014|17325            |258.36         |
|2015|16837            |261.11         |
|2016|16840            |270.9          |
|2017|16872            |268.78         |
|2018|87               |238.78         |
|2019|16737            |261.79         |
|2020|94               |250.87         |
|2021|65               |235.28         |
|2022|15192            |268.35         |
+----+-----------------+---------------+

Total de registros: 100049


In [ ]:
from pyspark.sql import functions as F

# Revisar los periodos disponibles en el dataset
tabla_periodos = (
    df_sin_duplicados
    .groupBy("periodo")
    .agg(
        F.count("*").alias("registros"),
        F.round(
            F.avg("punt_global"), 2
        ).alias("promedio_global")
    )
    .orderBy("periodo")
)

tabla_periodos.show(30, truncate=False)

print(
    "Total de registros:",
    df_sin_duplicados.count()
)

+-------+---------+---------------+
|periodo|registros|promedio_global|
+-------+---------+---------------+
|20142  |17325    |258.36         |
|20151  |299      |241.64         |
|20152  |16538    |261.46         |
|20162  |16840    |270.9          |
|20172  |16872    |268.78         |
|20181  |87       |238.78         |
|20194  |16737    |261.79         |
|20201  |94       |250.87         |
|20211  |65       |235.28         |
|20221  |120      |242.48         |
|20224  |15072    |268.56         |
+-------+---------+---------------+

Total de registros: 100049


In [ ]:
from pyspark.sql import functions as F

# Identificar periodos con mayor cobertura
periodos_principales = (
    tabla_periodos
    .filter(F.col("registros") >= 15000)
    .select("periodo")
)

# Crear un DataFrame para el análisis temporal principal
df_temporal = df_sin_duplicados.join(
    periodos_principales,
    on="periodo",
    how="inner"
)

# Generar año de presentación
df_temporal = df_temporal.withColumn(
    "anio",
    F.floor(F.col("periodo") / 10).cast("int")
)

# Calcular promedio por año
evolucion = (
    df_temporal
    .groupBy("anio")
    .agg(
        F.count("*").alias("estudiantes"),
        F.round(
            F.avg("punt_global"), 2
        ).alias("promedio_global")
    )
    .orderBy("anio")
)

print("EVOLUCIÓN EN PERIODOS DE MAYOR COBERTURA")
evolucion.show(truncate=False)

print(
    "Total de registros utilizados:",
    df_temporal.count()
)

EVOLUCIÓN EN PERIODOS DE MAYOR COBERTURA
+----+-----------+---------------+
|anio|estudiantes|promedio_global|
+----+-----------+---------------+
|2014|17325      |258.36         |
|2015|16538      |261.46         |
|2016|16840      |270.9          |
|2017|16872      |268.78         |
|2019|16737      |261.79         |
|2022|15072      |268.56         |
+----+-----------+---------------+

Total de registros utilizados: 99384


In [ ]:
from pyspark.sql import functions as F

# Clasificar los periodos según su cantidad de registros
tabla_cobertura = (
    tabla_periodos
    .withColumn(
        "cobertura",
        F.when(
            F.col("registros") >= 15000,
            "Mayor cobertura"
        ).otherwise("Menor cobertura")
    )
    .orderBy("periodo")
)

print("CLASIFICACIÓN DE PERIODOS")
tabla_cobertura.show(30, truncate=False)

# Verificar cantidad de periodos por categoría
tabla_cobertura.groupBy("cobertura").agg(
    F.count("*").alias("cantidad_periodos"),
    F.sum("registros").alias("total_registros")
).show(truncate=False)

CLASIFICACIÓN DE PERIODOS
+-------+---------+---------------+---------------+
|periodo|registros|promedio_global|cobertura      |
+-------+---------+---------------+---------------+
|20142  |17325    |258.36         |Mayor cobertura|
|20151  |299      |241.64         |Menor cobertura|
|20152  |16538    |261.46         |Mayor cobertura|
|20162  |16840    |270.9          |Mayor cobertura|
|20172  |16872    |268.78         |Mayor cobertura|
|20181  |87       |238.78         |Menor cobertura|
|20194  |16737    |261.79         |Mayor cobertura|
|20201  |94       |250.87         |Menor cobertura|
|20211  |65       |235.28         |Menor cobertura|
|20221  |120      |242.48         |Menor cobertura|
|20224  |15072    |268.56         |Mayor cobertura|
+-------+---------+---------------+---------------+

+---------------+-----------------+---------------+
|cobertura      |cantidad_periodos|total_registros|
+---------------+-----------------+---------------+
|Mayor cobertura|6                |99

In [ ]:
# Revisar la estructura del DataFrame original
print("ESTRUCTURA DEL DATASET ORIGINAL")
sdf.printSchema()

# Revisar las columnas disponibles
print("\nCOLUMNAS DEL DATASET:")
for columna in sdf.columns:
    print(columna)

# Comprobar la estructura del DataFrame limpio
print("\nESTRUCTURA DEL DATAFRAME LIMPIO:")
df_sin_duplicados.printSchema()

ESTRUCTURA DEL DATASET ORIGINAL
root
 |-- periodo: integer (nullable = true)
 |-- estu_tipodocumento: string (nullable = true)
 |-- estu_consecutivo: string (nullable = true)
 |-- cole_area_ubicacion: string (nullable = true)
 |-- cole_bilingue: string (nullable = true)
 |-- cole_calendario: string (nullable = true)
 |-- cole_caracter: string (nullable = true)
 |-- cole_cod_dane_establecimiento: long (nullable = true)
 |-- cole_cod_dane_sede: long (nullable = true)
 |-- cole_cod_depto_ubicacion: integer (nullable = true)
 |-- cole_cod_mcpio_ubicacion: integer (nullable = true)
 |-- cole_codigo_icfes: integer (nullable = true)
 |-- cole_depto_ubicacion: string (nullable = true)
 |-- cole_genero: string (nullable = true)
 |-- cole_jornada: string (nullable = true)
 |-- cole_mcpio_ubicacion: string (nullable = true)
 |-- cole_naturaleza: string (nullable = true)
 |-- cole_nombre_establecimiento: string (nullable = true)
 |-- cole_nombre_sede: string (nullable = true)
 |-- cole_sede_princi

In [14]:
# Seleccionar las columnas necesarias para el análisis

columnas_analisis = [
    "periodo",
    "cole_cod_depto_ubicacion",
    "cole_cod_mcpio_ubicacion",
    "cole_mcpio_ubicacion",
    "cole_naturaleza",
    "cole_area_ubicacion",
    "cole_jornada",
    "fami_estratovivienda",
    "fami_educacionmadre",
    "fami_tieneinternet",
    "punt_global",
    "punt_matematicas",
    "punt_lectura_critica",
    "punt_c_naturales",
    "punt_sociales_ciudadanas",
    "punt_ingles"
]

# Crear una copia con las columnas seleccionadas
df_analisis = df_sin_duplicados.select(
    *columnas_analisis
)

print("COLUMNAS DEL DATASET ANALÍTICO")
print("Total de columnas:", len(df_analisis.columns))

print("\nTOTAL DE REGISTROS")
print(df_analisis.count())

print("\nESTRUCTURA")
df_analisis.printSchema()

COLUMNAS DEL DATASET ANALÍTICO
Total de columnas: 16

TOTAL DE REGISTROS
100049

ESTRUCTURA
root
 |-- periodo: integer (nullable = true)
 |-- cole_cod_depto_ubicacion: integer (nullable = true)
 |-- cole_cod_mcpio_ubicacion: integer (nullable = true)
 |-- cole_mcpio_ubicacion: string (nullable = true)
 |-- cole_naturaleza: string (nullable = true)
 |-- cole_area_ubicacion: string (nullable = true)
 |-- cole_jornada: string (nullable = true)
 |-- fami_estratovivienda: string (nullable = true)
 |-- fami_educacionmadre: string (nullable = true)
 |-- fami_tieneinternet: string (nullable = true)
 |-- punt_global: integer (nullable = true)
 |-- punt_matematicas: double (nullable = true)
 |-- punt_lectura_critica: integer (nullable = true)
 |-- punt_c_naturales: integer (nullable = true)
 |-- punt_sociales_ciudadanas: integer (nullable = true)
 |-- punt_ingles: double (nullable = true)



In [15]:
# Revisar categorías antes de normalizarlas

columnas_categoricas = [
    "cole_naturaleza",
    "fami_estratovivienda",
    "fami_educacionmadre",
    "fami_tieneinternet"
]

for columna in columnas_categoricas:

    print("\n" + "=" * 60)
    print("COLUMNA:", columna)
    print("=" * 60)

    (
        df_analisis
        .groupBy(columna)
        .count()
        .orderBy("count", ascending=False)
        .show(50, truncate=False)
    )


COLUMNA: cole_naturaleza
+---------------+-----+
|cole_naturaleza|count|
+---------------+-----+
|OFICIAL        |81513|
|NO OFICIAL     |18536|
+---------------+-----+


COLUMNA: fami_estratovivienda
+--------------------+-----+
|fami_estratovivienda|count|
+--------------------+-----+
|Estrato 2           |47336|
|Estrato 1           |31872|
|Estrato 3           |13769|
|Estrato 4           |2772 |
|NULL                |2642 |
|Estrato 5           |914  |
|Sin Estrato         |569  |
|Estrato 6           |175  |
+--------------------+-----+


COLUMNA: fami_educacionmadre
+------------------------------------+-----+
|fami_educacionmadre                 |count|
+------------------------------------+-----+
|Secundaria (Bachillerato) completa  |23564|
|Primaria incompleta                 |18248|
|Primaria completa                   |17026|
|Secundaria (Bachillerato) incompleta|13659|
|Educación profesional completa      |9303 |
|Técnica o tecnológica completa      |6432 |
|Postgrado    

In [21]:
from pyspark.sql import functions as F

# =====================================================
# LIMPIEZA DEFINITIVA - PRUEBAS SABER 11 BOYACÁ
# =====================================================

# 1. Filtrar periodos y eliminar filas idénticas
df = (
    sdf
    .filter(F.col("periodo") >= 20142)
    .dropDuplicates()
)

# 2. Normalizar textos
for columna, tipo in df.dtypes:
    if tipo == "string":
        df = df.withColumn(
            columna,
            F.upper(F.trim(F.col(columna)))
        )

# 3. Validar los rangos de los puntajes
pruebas = [
    "punt_ingles",
    "punt_matematicas",
    "punt_lectura_critica",
    "punt_sociales_ciudadanas",
    "punt_c_naturales"
]

for columna in pruebas:
    df = df.withColumn(
        columna,
        F.when(
            (F.col(columna) < 0) |
            (F.col(columna) > 100),
            F.lit(None)
        ).otherwise(F.col(columna))
    )

df = df.withColumn(
    "punt_global",
    F.when(
        (F.col("punt_global") < 0) |
        (F.col("punt_global") > 500),
        F.lit(None)
    ).otherwise(F.col("punt_global"))
)

# 4. Normalizar valores faltantes
columnas_familiares = [
    c for c in df.columns
    if c.startswith("fami_")
]

for columna in columnas_familiares + ["cole_bilingue"]:
    df = df.withColumn(
        columna,
        F.when(
            F.col(columna).isNull() |
            (F.col(columna) == ""),
            F.lit("NO INFORMADO")
        ).otherwise(F.col(columna))
    )

# 5. Crear año y ciclo
df = (
    df
    .withColumn(
        "anio",
        F.floor(F.col("periodo") / 10).cast("int")
    )
    .withColumn(
        "ciclo",
        (F.col("periodo") % 10).cast("int")
    )
)

# 6. Identificar registros con todos los puntajes en cero
condicion_ceros = F.col("punt_global") == 0

for columna in pruebas:
    condicion_ceros = (
        condicion_ceros &
        (F.col(columna) == 0)
    )

df = df.withColumn(
    "puntajes_todos_cero",
    F.when(
        condicion_ceros,
        True
    ).otherwise(False)
)

# 7. Calcular edad aproximada
fecha = F.coalesce(
    F.to_date("estu_fechanacimiento", "dd/MM/yyyy"),
    F.to_date("estu_fechanacimiento", "yyyy-MM-dd"),
    F.to_date("estu_fechanacimiento", "MM/dd/yyyy")
)

df = df.withColumn(
    "edad",
    F.col("anio") - F.year(fecha)
)

df = df.withColumn(
    "edad",
    F.when(
        (F.col("edad") < 10) |
        (F.col("edad") > 80),
        F.lit(None)
    ).otherwise(F.col("edad"))
)

# 8. Seudonimizar identificadores
df = (
    df
    .withColumn(
        "id_anonimo",
        F.sha2(
            F.col("estu_consecutivo"),
            256
        )
    )
    .drop(
        "estu_consecutivo",
        "estu_fechanacimiento"
    )
)

# 9. Verificar resultados
print("REGISTROS DESPUÉS DE LA LIMPIEZA:", df.count())

print("COLUMNAS DESPUÉS DE LA LIMPIEZA:", len(df.columns))

print("\nREGISTROS CON TODOS LOS PUNTAJES EN CERO:")
df.groupBy("puntajes_todos_cero").count().show()

print("\nPROMEDIO GLOBAL:")
df.agg(
    F.round(
        F.avg("punt_global"),
        2
    ).alias("promedio_global")
).show()

REGISTROS DESPUÉS DE LA LIMPIEZA: 100049
COLUMNAS DESPUÉS DE LA LIMPIEZA: 54

REGISTROS CON TODOS LOS PUNTAJES EN CERO:
+-------------------+------+
|puntajes_todos_cero| count|
+-------------------+------+
|              false|100048|
|               true|     1|
+-------------------+------+


PROMEDIO GLOBAL:
+---------------+
|promedio_global|
+---------------+
|         264.74|
+---------------+



In [22]:
from pyspark.sql import functions as F
import pandas as pd

# =====================================================
# CONTROL DE CALIDAD Y CAPA PLATA
# =====================================================

# 1. Comparar indicadores antes y después
def resumen(d, identificador):
    total = d.count()

    return {
        "Filas": total,
        "Duplicados exactos": total - d.dropDuplicates().count(),
        "Duplicados por ID": total - d.dropDuplicates([identificador]).count(),
        "Nulos puntaje global": d.filter(
            F.col("punt_global").isNull()
        ).count(),
        "Periodos distintos": d.select("periodo").distinct().count()
    }

antes = resumen(sdf, "estu_consecutivo")
despues = resumen(df, "id_anonimo")

metricas = pd.DataFrame({
    "Antes": antes,
    "Después": despues
})

print("COMPARACIÓN DE CALIDAD")
display(metricas)

# 2. Seleccionar variables necesarias para la capa Plata
columnas_plata = [
    "periodo",
    "anio",
    "ciclo",
    "cole_cod_depto_ubicacion",
    "cole_cod_mcpio_ubicacion",
    "cole_mcpio_ubicacion",
    "cole_naturaleza",
    "cole_area_ubicacion",
    "cole_jornada",
    "fami_estratovivienda",
    "fami_educacionmadre",
    "fami_tieneinternet",
    "punt_global",
    "punt_matematicas",
    "punt_lectura_critica",
    "punt_c_naturales",
    "punt_sociales_ciudadanas",
    "punt_ingles",
    "puntajes_todos_cero"
]

df_plata = df.select(*columnas_plata)

# 3. Guardar capa Plata en Parquet
ruta_plata = "data/clean/saber11_boyaca_limpio.parquet"

df_plata.write.mode("overwrite").parquet(ruta_plata)

# 4. Verificar el archivo guardado
verificacion = spark.read.parquet(ruta_plata)

print("\nVERIFICACIÓN DE CAPA PLATA")
print("Registros:", verificacion.count())
print("Columnas:", len(verificacion.columns))

print("\nREGISTROS POR BANDERA DE CALIDAD")
verificacion.groupBy("puntajes_todos_cero").count().show()

print("\nCATEGORÍAS DE ESTRATO")
verificacion.groupBy("fami_estratovivienda").count().show(
    truncate=False
)

COMPARACIÓN DE CALIDAD


,Antes,Después
Filas,215234,100049
Duplicados exactos,38576,0
Duplicados por ID,40415,0
Nulos puntaje global,83150,0
Periodos distintos,20,11



VERIFICACIÓN DE CAPA PLATA
Registros: 100049
Columnas: 19

REGISTROS POR BANDERA DE CALIDAD
+-------------------+------+
|puntajes_todos_cero| count|
+-------------------+------+
|               true|     1|
|              false|100048|
+-------------------+------+


CATEGORÍAS DE ESTRATO
+--------------------+-----+
|fami_estratovivienda|count|
+--------------------+-----+
|ESTRATO 2           |47336|
|SIN ESTRATO         |569  |
|ESTRATO 5           |914  |
|ESTRATO 6           |175  |
|ESTRATO 3           |13769|
|ESTRATO 4           |2772 |
|NO INFORMADO        |2642 |
|ESTRATO 1           |31872|
+--------------------+-----+



In [25]:
from pyspark.sql import functions as F

# =====================================================
# ANÁLISIS DE LAS 4 PREGUNTAS DEL PROYECTO
# =====================================================

# Utilizar la capa Plata validada
df_analitico = spark.read.parquet(
    "data/clean/saber11_boyaca_limpio.parquet"
)

df_analitico.createOrReplaceTempView("saber")

# =====================================================
# P1. EVOLUCIÓN DEL PUNTAJE GLOBAL POR AÑO
# =====================================================

# Identificar periodos con al menos 15.000 registros
periodos_principales = (
    df_analitico
    .groupBy("periodo")
    .count()
    .filter(F.col("count") >= 15000)
    .select("periodo")
)

df_temporal = df_analitico.join(
    periodos_principales,
    on="periodo",
    how="inner"
)

df_temporal.createOrReplaceTempView("saber_temporal")

q1 = spark.sql("""
    SELECT
        anio,
        COUNT(*) AS estudiantes,
        ROUND(AVG(punt_global), 2) AS prom_global
    FROM saber_temporal
    GROUP BY anio
    ORDER BY anio
""")

print("P1. EVOLUCIÓN DEL PUNTAJE GLOBAL")
print("Periodos de mayor cobertura dentro del dataset")
q1.show(truncate=False)

print("Registros utilizados en P1:", df_temporal.count())

# =====================================================
# P2. COLEGIOS OFICIALES VS NO OFICIALES
# =====================================================

q2 = (
    df_analitico
    .groupBy("cole_naturaleza")
    .agg(
        F.count("*").alias("estudiantes"),
        F.round(F.avg("punt_global"), 2).alias("prom_global")
    )
    .orderBy("cole_naturaleza")
)

print("\nP2. RESULTADOS SEGÚN NATURALEZA DEL COLEGIO")
q2.show(truncate=False)

# =====================================================
# P3. FACTORES SOCIOECONÓMICOS
# =====================================================

variables_socioeconomicas = [
    "fami_estratovivienda",
    "fami_tieneinternet",
    "fami_educacionmadre"
]

resultados_p3 = {}

for columna in variables_socioeconomicas:

    resultado = (
        df_analitico
        .groupBy(columna)
        .agg(
            F.count("*").alias("estudiantes"),
            F.round(F.avg("punt_global"), 2).alias("prom_global")
        )
        .orderBy("prom_global")
    )

    resultados_p3[columna] = resultado

    print("\nP3. VARIABLE:", columna)
    resultado.show(30, truncate=False)

# =====================================================
# P4. MUNICIPIOS CON MAYORES Y MENORES PROMEDIOS
# =====================================================

# 1. Unificar nombres usando el código del municipio
# Elegir la variante más frecuente para cada código

frecuencias_nombres = (
    df_analitico
    .groupBy(
        "cole_cod_mcpio_ubicacion",
        "cole_mcpio_ubicacion"
    )
    .count()
)

from pyspark.sql.window import Window

ventana = Window.partitionBy(
    "cole_cod_mcpio_ubicacion"
).orderBy(
    F.desc("count"),
    F.asc("cole_mcpio_ubicacion")
)

nombres_unificados = (
    frecuencias_nombres
    .withColumn(
        "fila",
        F.row_number().over(ventana)
    )
    .filter(F.col("fila") == 1)
    .select(
        "cole_cod_mcpio_ubicacion",
        F.col("cole_mcpio_ubicacion").alias("municipio")
    )
)

# 2. Agrupar por código municipal
estadisticas_municipales = (
    df_analitico
    .groupBy("cole_cod_mcpio_ubicacion")
    .agg(
        F.count("*").alias("estudiantes"),
        F.round(
            F.avg("punt_global"), 2
        ).alias("prom_global")
    )
)

# 3. Incorporar el nombre unificado
q4 = (
    estadisticas_municipales
    .join(
        nombres_unificados,
        on="cole_cod_mcpio_ubicacion",
        how="left"
    )
    .filter(F.col("estudiantes") >= 100)
    .select(
        "cole_cod_mcpio_ubicacion",
        "municipio",
        "estudiantes",
        "prom_global"
    )
)

# 4. Mostrar resultados
print("P4. MUNICIPIOS CON MAYORES PROMEDIOS")

q4.orderBy(
    F.desc("prom_global")
).show(10, truncate=False)

print("\nP4. MUNICIPIOS CON MENORES PROMEDIOS")

q4.orderBy(
    F.asc("prom_global")
).show(10, truncate=False)

print("\nTOTAL DE MUNICIPIOS ANALIZADOS:", q4.count())

print(
    "TOTAL DE CÓDIGOS MUNICIPALES:",
    df_analitico.select(
        "cole_cod_mcpio_ubicacion"
    ).distinct().count()
)

P1. EVOLUCIÓN DEL PUNTAJE GLOBAL
Periodos de mayor cobertura dentro del dataset
+----+-----------+-----------+
|anio|estudiantes|prom_global|
+----+-----------+-----------+
|2014|17325      |258.36     |
|2015|16538      |261.46     |
|2016|16840      |270.9      |
|2017|16872      |268.78     |
|2019|16737      |261.79     |
|2022|15072      |268.56     |
+----+-----------+-----------+

Registros utilizados en P1: 99384

P2. RESULTADOS SEGÚN NATURALEZA DEL COLEGIO
+---------------+-----------+-----------+
|cole_naturaleza|estudiantes|prom_global|
+---------------+-----------+-----------+
|NO OFICIAL     |18536      |285.07     |
|OFICIAL        |81513      |260.12     |
+---------------+-----------+-----------+


P3. VARIABLE: fami_estratovivienda
+--------------------+-----------+-----------+
|fami_estratovivienda|estudiantes|prom_global|
+--------------------+-----------+-----------+
|SIN ESTRATO         |569        |239.05     |
|ESTRATO 1           |31872      |249.94     |
|NO IN

In [24]:
from pyspark.sql import functions as F

# =====================================================
# DIAGNÓSTICO DE CÓDIGOS Y NOMBRES DE MUNICIPIOS
# =====================================================

print("CÓDIGOS DE MUNICIPIO DISTINTOS:")

df_analitico.select(
    "cole_cod_mcpio_ubicacion"
).distinct().count()

print(
    df_analitico.select(
        "cole_cod_mcpio_ubicacion"
    ).distinct().count()
)

print("\nNOMBRES DE MUNICIPIO DISTINTOS:")

print(
    df_analitico.select(
        "cole_mcpio_ubicacion"
    ).distinct().count()
)

print("\nCÓDIGOS CON MÁS DE UN NOMBRE:")

variantes = (
    df_analitico
    .groupBy("cole_cod_mcpio_ubicacion")
    .agg(
        F.countDistinct(
            "cole_mcpio_ubicacion"
        ).alias("cantidad_nombres"),
        F.collect_set(
            "cole_mcpio_ubicacion"
        ).alias("nombres")
    )
    .filter(F.col("cantidad_nombres") > 1)
    .orderBy(F.desc("cantidad_nombres"))
)

variantes.show(30, truncate=False)

print("\nCÓDIGOS NULOS:")

df_analitico.filter(
    F.col("cole_cod_mcpio_ubicacion").isNull()
).count()

CÓDIGOS DE MUNICIPIO DISTINTOS:
123

NOMBRES DE MUNICIPIO DISTINTOS:
174

CÓDIGOS CON MÁS DE UN NOMBRE:
+------------------------+----------------+------------------------------------+
|cole_cod_mcpio_ubicacion|cantidad_nombres|nombres                             |
+------------------------+----------------+------------------------------------+
|15296                   |2               |[GÁMEZA, GAMEZA]                    |
|15332                   |2               |[GÜICÁN DE LA SIERRA, GUICAN]       |
|15599                   |2               |[RAMIRIQUÍ, RAMIRIQUI]              |
|15696                   |2               |[SANTA SOFIA, SANTA SOFÍA]          |
|15835                   |2               |[TURMEQUÉ, TURMEQUE]                |
|15638                   |2               |[SÁCHICA, SACHICA]                  |
|15223                   |2               |[CUBARÁ, CUBARA]                    |
|15621                   |2               |[RONDON, RONDÓN]                    |
|1576

0

In [26]:
import os
import pandas as pd
from pyspark.sql import functions as F

# =====================================================
# EXPORTACIÓN DE DATOS PARA DASHBOARD STREAMLIT
# =====================================================

# 1. Incorporar nombres municipales unificados
df_dashboard = (
    df_analitico
    .drop("cole_mcpio_ubicacion")
    .join(
        nombres_unificados,
        on="cole_cod_mcpio_ubicacion",
        how="left"
    )
)

# 2. Seleccionar columnas necesarias para el dashboard
columnas_dashboard = [
    "periodo",
    "anio",
    "cole_cod_mcpio_ubicacion",
    "municipio",
    "cole_naturaleza",
    "cole_area_ubicacion",
    "fami_estratovivienda",
    "fami_educacionmadre",
    "fami_tieneinternet",
    "punt_global",
    "punt_matematicas",
    "punt_lectura_critica",
    "punt_c_naturales",
    "punt_sociales_ciudadanas",
    "punt_ingles",
    "puntajes_todos_cero"
]

df_dashboard = df_dashboard.select(*columnas_dashboard)

# 3. Crear carpeta de salida
os.makedirs("data", exist_ok=True)

# 4. Exportar CSV comprimido
ruta_dashboard = "data/saber11_dashboard.csv.gz"

df_dashboard.toPandas().to_csv(
    ruta_dashboard,
    index=False,
    compression="gzip"
)

# 5. Verificar archivo exportado
verificacion = pd.read_csv(ruta_dashboard)

print("ARCHIVO GENERADO CORRECTAMENTE")
print("Ruta:", ruta_dashboard)
print("Registros:", len(verificacion))
print("Columnas:", len(verificacion.columns))
print("Municipios diferentes:", verificacion["municipio"].nunique())

ARCHIVO GENERADO CORRECTAMENTE
Ruta: data/saber11_dashboard.csv.gz
Registros: 100049
Columnas: 16
Municipios diferentes: 123


In [27]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

# =====================================================
# CONFIGURACIÓN
# =====================================================

st.set_page_config(
    page_title="Saber 11 Boyacá",
    page_icon="📊",
    layout="wide"
)

# =====================================================
# CARGAR DATOS
# =====================================================

@st.cache_data
def cargar_datos():
    return pd.read_csv(
        "data/saber11_dashboard.csv.gz",
        dtype={"cole_cod_mcpio_ubicacion": "string"}
    )

df = cargar_datos()

st.title("📊 Análisis de las Pruebas Saber 11 en Boyacá")
st.caption(
    "Análisis descriptivo de registros disponibles "
    "entre los periodos 2014-2 y 2022-4."
)

# =====================================================
# FILTROS
# =====================================================

st.sidebar.header("Filtros interactivos")

anios = sorted(df["anio"].dropna().unique().tolist())

sel_anios = st.sidebar.multiselect(
    "Año",
    anios,
    default=anios
)

naturalezas = sorted(
    df["cole_naturaleza"].dropna().unique().tolist()
)

sel_naturaleza = st.sidebar.multiselect(
    "Naturaleza del colegio",
    naturalezas,
    default=naturalezas
)

municipios = sorted(
    df["municipio"].dropna().unique().tolist()
)

sel_municipios = st.sidebar.multiselect(
    "Municipio",
    municipios,
    default=municipios
)

# Aplicar filtros
f = df[
    df["anio"].isin(sel_anios)
    & df["cole_naturaleza"].isin(sel_naturaleza)
    & df["municipio"].isin(sel_municipios)
].copy()

if f.empty:
    st.warning("No existen registros para los filtros seleccionados.")
    st.stop()

# =====================================================
# INDICADORES GENERALES
# =====================================================

st.subheader("Indicadores generales")

c1, c2, c3, c4 = st.columns(4)

c1.metric(
    "Registros analizados",
    f"{len(f):,}"
)

c2.metric(
    "Puntaje global promedio",
    f"{f['punt_global'].mean():.2f}"
)

c3.metric(
    "Municipios",
    f["cole_cod_mcpio_ubicacion"].nunique()
)

c4.metric(
    "Colegios oficiales",
    f"{(f['cole_naturaleza'] == 'OFICIAL').sum():,}"
)

# =====================================================
# P1. EVOLUCIÓN TEMPORAL
# =====================================================

st.subheader("P1. Evolución del puntaje global")

# Periodos principales identificados en el análisis
periodos_principales = [
    20142, 20152, 20162,
    20172, 20194, 20224
]

temporal = f[
    f["periodo"].isin(periodos_principales)
]

por_anio = (
    temporal
    .groupby("anio", as_index=False)
    .agg(
        promedio=("punt_global", "mean"),
        estudiantes=("punt_global", "size")
    )
    .sort_values("anio")
)

fig1 = px.line(
    por_anio,
    x="anio",
    y="promedio",
    markers=True,
    hover_data=["estudiantes"],
    title="Puntaje global promedio por año"
)

fig1.update_xaxes(dtick=1)

st.plotly_chart(fig1, use_container_width=True)

st.caption(
    "Se muestran los seis periodos con al menos "
    "15.000 registros en el dataset original. "
    "Este criterio no garantiza representatividad poblacional."
)

# =====================================================
# P2. OFICIAL VS NO OFICIAL
# =====================================================

st.subheader("P2. Naturaleza del colegio")

por_naturaleza = (
    f.groupby("cole_naturaleza", as_index=False)
    .agg(
        promedio=("punt_global", "mean"),
        estudiantes=("punt_global", "size")
    )
)

fig2 = px.bar(
    por_naturaleza,
    x="cole_naturaleza",
    y="promedio",
    color="cole_naturaleza",
    hover_data=["estudiantes"],
    title="Puntaje promedio según naturaleza del colegio"
)

st.plotly_chart(fig2, use_container_width=True)

# =====================================================
# P3. FACTORES SOCIOECONÓMICOS
# =====================================================

st.subheader("P3. Factores socioeconómicos")

col1, col2 = st.columns(2)

por_estrato = (
    f.groupby("fami_estratovivienda", as_index=False)
    .agg(
        promedio=("punt_global", "mean"),
        estudiantes=("punt_global", "size")
    )
)

fig3 = px.bar(
    por_estrato,
    x="fami_estratovivienda",
    y="promedio",
    hover_data=["estudiantes"],
    title="Puntaje promedio por estrato"
)

col1.plotly_chart(fig3, use_container_width=True)

por_internet = (
    f.groupby("fami_tieneinternet", as_index=False)
    .agg(
        promedio=("punt_global", "mean"),
        estudiantes=("punt_global", "size")
    )
)

fig4 = px.bar(
    por_internet,
    x="fami_tieneinternet",
    y="promedio",
    hover_data=["estudiantes"],
    title="Puntaje promedio según acceso a internet"
)

col2.plotly_chart(fig4, use_container_width=True)

# =====================================================
# P4. MUNICIPIOS
# =====================================================

st.subheader("P4. Comparación municipal")

municipal = (
    f.groupby(
        ["cole_cod_mcpio_ubicacion", "municipio"],
        as_index=False
    )
    .agg(
        estudiantes=("punt_global", "size"),
        promedio=("punt_global", "mean")
    )
)

# Aplicar mínimo de registros
municipal = municipal[
    municipal["estudiantes"] >= 100
]

col3, col4 = st.columns(2)

top10 = municipal.nlargest(10, "promedio")

fig5 = px.bar(
    top10.sort_values("promedio"),
    x="promedio",
    y="municipio",
    orientation="h",
    hover_data=["estudiantes"],
    title="10 municipios con mayores promedios"
)

col3.plotly_chart(fig5, use_container_width=True)

bottom10 = municipal.nsmallest(10, "promedio")

fig6 = px.bar(
    bottom10.sort_values("promedio", ascending=False),
    x="promedio",
    y="municipio",
    orientation="h",
    hover_data=["estudiantes"],
    title="10 municipios con menores promedios"
)

col4.plotly_chart(fig6, use_container_width=True)

st.caption(
    "Los rankings incluyen únicamente municipios "
    "con al menos 100 registros después de aplicar "
    "los filtros seleccionados."
)

# =====================================================
# CALIDAD Y LIMITACIONES
# =====================================================

st.subheader("Calidad y limitaciones de los datos")

st.write(
    "Los resultados describen los registros disponibles "
    "en el conjunto de datos analizado. "
    "No deben interpretarse como estimaciones causales "
    "ni necesariamente como resultados representativos "
    "de todos los estudiantes de Boyacá."
)

st.write(
    "Registros con todos los puntajes en cero:",
    int(f["puntajes_todos_cero"].sum())
)

st.success("Dashboard cargado correctamente.")

Writing app.py


In [29]:
%pip install -q streamlit plotly

In [31]:
import subprocess
import time

# Iniciar Streamlit en segundo plano
proceso = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Estado del proceso:", proceso.poll())
print("\nÚltimos mensajes de Streamlit:")

with open("streamlit.log", "r") as archivo:
    print(archivo.read()[-2000:])

Estado del proceso: None

Últimos mensajes de Streamlit:


2026-11-19 16:33:04.229 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.255.131.106:8501




In [32]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=900
)

<IPython.core.display.Javascript object>

In [34]:
from google.colab import output

url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("Enlace del dashboard:")
print(url)

Enlace del dashboard:
https://8501-m-s-usc1b11-2u7coh777gxwa-b.us-central1.prod.r.colab.dev


In [36]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

cloudflared version 2026.11.1 (built 2026-11-17-15:04 UTC)


In [37]:
import subprocess
import time
import re

# Iniciar túnel de Cloudflare
tunel = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501",
        "--no-autoupdate"
    ],
    stdout=open("cloudflared.log", "w"),
    stderr=subprocess.STDOUT
)

# Esperar a que se genere el enlace
time.sleep(10)

# Leer resultados
with open("cloudflared.log", "r") as archivo:
    contenido = archivo.read()

enlaces = re.findall(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    contenido
)

if enlaces:
    print("ENLACE TEMPORAL DEL DASHBOARD:")
    print(enlaces[0])
else:
    print("Todavía no se encontró el enlace.")
    print(contenido[-2000:])

ENLACE TEMPORAL DEL DASHBOARD:
https://walking-windsor-rss-guild.trycloudflare.com


In [40]:
from pyspark.sql import functions as F

# =====================================================
# ANÁLISIS COMPLEMENTARIO: CORRELACIÓN DE PEARSON
# =====================================================

# Trabajar sobre la capa limpia sin modificarla
df_correlacion = df_analitico

# 1. Convertir estrato a valor numérico
df_correlacion = df_correlacion.withColumn(
    "estrato_num",
    F.when(
        F.col("fami_estratovivienda").rlike(r"^ESTRATO [1-6]$"),
        F.regexp_extract(
            "fami_estratovivienda",
            r"([1-6])",
            1
        ).cast("double")
    )
)

# 2. Codificar acceso a internet
df_correlacion = df_correlacion.withColumn(
    "internet_num",
    F.when(F.col("fami_tieneinternet") == "SI", 1.0)
     .when(F.col("fami_tieneinternet") == "NO", 0.0)
)

# 3. Codificar naturaleza del colegio
df_correlacion = df_correlacion.withColumn(
    "no_oficial",
    F.when(F.col("cole_naturaleza") == "NO OFICIAL", 1.0)
     .when(F.col("cole_naturaleza") == "OFICIAL", 0.0)
)

# 4. Calcular correlaciones
variables = [
    "estrato_num",
    "internet_num",
    "no_oficial"
]

print("CORRELACIONES CON EL PUNTAJE GLOBAL")
print("=" * 55)

for columna in variables:

    datos_validos = df_correlacion.filter(
        F.col(columna).isNotNull()
        & F.col("punt_global").isNotNull()
    )

    cantidad = datos_validos.count()

    correlacion = datos_validos.stat.corr(
        "punt_global",
        columna
    )

    print(
        f"{columna}: "
        f"correlación = {correlacion:.3f} | "
        f"registros = {cantidad}"
    )

CORRELACIONES CON EL PUNTAJE GLOBAL
estrato_num: correlación = 0.277 | registros = 96838
internet_num: correlación = 0.287 | registros = 97585
no_oficial: correlación = 0.212 | registros = 100049


In [41]:
import pandas as pd
import plotly.express as px

# =====================================================
# ANÁLISIS ESTADÍSTICO DESCRIPTIVO
# =====================================================

# Cargar datos definitivos del dashboard
datos_estadisticos = pd.read_csv(
    "data/saber11_dashboard.csv.gz"
)

# 1. Estadística descriptiva con PySpark
columnas_puntajes = [
    "punt_global",
    "punt_matematicas",
    "punt_lectura_critica",
    "punt_ingles",
    "punt_sociales_ciudadanas",
    "punt_c_naturales"
]

print("ESTADÍSTICAS DESCRIPTIVAS DE LOS PUNTAJES")

df_analitico.select(columnas_puntajes).summary(
    "count", "mean", "stddev",
    "min", "25%", "50%", "75%", "max"
).show()

# 2. Distribución del puntaje global
fig1 = px.histogram(
    datos_estadisticos,
    x="punt_global",
    nbins=50,
    title="Distribución del puntaje global - Saber 11 Boyacá"
)

fig1.show()

# 3. Comparación entre colegios oficiales y no oficiales
fig2 = px.box(
    datos_estadisticos,
    x="cole_naturaleza",
    y="punt_global",
    title="Distribución del puntaje global según naturaleza del colegio"
)

fig2.show()

ESTADÍSTICAS DESCRIPTIVAS DE LOS PUNTAJES
+-------+------------------+------------------+--------------------+------------------+------------------------+------------------+
|summary|       punt_global|  punt_matematicas|punt_lectura_critica|       punt_ingles|punt_sociales_ciudadanas|  punt_c_naturales|
+-------+------------------+------------------+--------------------+------------------+------------------------+------------------+
|  count|            100049|            100049|              100049|            100047|                  100049|            100049|
|   mean|264.74132674989255| 54.06531799418285|  53.374376555487814| 51.63191300088958|       51.69440973922778|53.097462243500686|
| stddev|  45.6881262585804|11.237403055862588|   9.676324225506766|10.681404672375793|       10.89338924245628|  9.54341079727822|
|    min|                 0|               0.0|                   0|               0.0|                       0|                 0|
|    25%|               232|      

In [42]:
import os
import shutil
import pandas as pd
from pyspark.sql import functions as F

# =====================================================
# ARQUITECTURA MEDALLION: BRONCE, PLATA Y ORO
# =====================================================

# 1. Crear carpetas
for capa in ["bronce", "plata", "oro"]:
    os.makedirs(f"data/{capa}", exist_ok=True)

# 2. BRONCE: conservar archivo original
origen = "data/raw/saber11_boyaca.csv"
destino = "data/bronce/saber11_boyaca_crudo.csv"

if os.path.exists(origen):
    shutil.copy2(origen, destino)
    print("Capa Bronce creada correctamente.")
else:
    print("Aviso: no se encontró el CSV original en", origen)

# 3. PLATA: guardar datos limpios y validados
df_analitico.write.mode("overwrite").parquet(
    "data/plata/saber11_boyaca_limpio.parquet"
)

print("Capa Plata creada correctamente.")

# 4. ORO: generar tablas agregadas

def guardar_oro(nombre, datos):
    ruta = f"data/oro/{nombre}"
    datos.write.mode("overwrite").parquet(ruta)
    print(nombre, "->", datos.count(), "filas")

def agregar_promedio(*columnas):
    return df_analitico.groupBy(*columnas).agg(
        F.count("*").alias("estudiantes"),
        F.round(F.avg("punt_global"), 2).alias("prom_global")
    )

# P1. Evolución temporal
guardar_oro(
    "por_anio",
    agregar_promedio("periodo", "anio")
)

# P2. Naturaleza del colegio
guardar_oro(
    "por_naturaleza",
    agregar_promedio("cole_naturaleza")
)

# P3. Factores socioeconómicos
guardar_oro(
    "por_estrato",
    agregar_promedio("fami_estratovivienda")
)

guardar_oro(
    "por_internet",
    agregar_promedio("fami_tieneinternet")
)

guardar_oro(
    "por_educacion_madre",
    agregar_promedio("fami_educacionmadre")
)

# P4. Municipios: utilizar códigos únicos
guardar_oro(
    "por_municipio",
    agregar_promedio("cole_cod_mcpio_ubicacion")
)

# 5. DICCIONARIO DE DATOS
# Evitamos publicar ejemplos de registros individuales.

total = df_analitico.count()

conteo_nulos = df_analitico.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in df_analitico.columns
]).collect()[0].asDict()

diccionario = pd.DataFrame([
    {
        "columna": columna,
        "tipo": tipo,
        "nulos": conteo_nulos[columna],
        "porcentaje_nulos": round(
            conteo_nulos[columna] / total * 100, 2
        ),
        "descripcion": ""
    }
    for columna, tipo in df_analitico.dtypes
])

diccionario.to_csv(
    "data/oro/diccionario_datos.csv",
    index=False,
    encoding="utf-8-sig"
)

display(diccionario)

# 6. VERIFICACIÓN DE PUNTAJES EXTREMOS
print("Puntaje global igual a 0:",
      df_analitico.filter(F.col("punt_global") == 0).count())

print("Puntaje global menor que 100:",
      df_analitico.filter(F.col("punt_global") < 100).count())

print("Registros con todos los puntajes en cero:",
      df_analitico.filter(
          F.col("puntajes_todos_cero") == True
      ).count())

print("Arquitectura Medallion generada.")

Capa Bronce creada correctamente.
Capa Plata creada correctamente.
por_anio -> 11 filas
por_naturaleza -> 2 filas
por_estrato -> 8 filas
por_internet -> 3 filas
por_educacion_madre -> 13 filas
por_municipio -> 123 filas


,columna,tipo,nulos,porcentaje_nulos,descripcion
0,periodo,int,0,0.0,
1,anio,int,0,0.0,
2,ciclo,int,0,0.0,
3,cole_cod_depto_ubicacion,int,0,0.0,
4,cole_cod_mcpio_ubicacion,int,0,0.0,
5,cole_mcpio_ubicacion,string,0,0.0,
6,cole_naturaleza,string,0,0.0,
7,cole_area_ubicacion,string,0,0.0,
8,cole_jornada,string,0,0.0,
9,fami_estratovivienda,string,0,0.0,


Puntaje global igual a 0: 1
Puntaje global menor que 100: 9
Registros con todos los puntajes en cero: 1
Arquitectura Medallion generada.


In [43]:
import pandas as pd

# =====================================================
# DICCIONARIO DE DATOS DEFINITIVO
# =====================================================

descripciones = {
    "periodo": "Periodo de aplicación de la prueba Saber 11.",
    "anio": "Año de aplicación, derivado del periodo.",
    "ciclo": "Ciclo de aplicación, derivado del periodo.",
    "cole_cod_depto_ubicacion": "Código DANE del departamento donde se ubica el colegio.",
    "cole_cod_mcpio_ubicacion": "Código DANE del municipio donde se ubica el colegio.",
    "cole_mcpio_ubicacion": "Nombre del municipio donde se ubica el colegio.",
    "cole_naturaleza": "Naturaleza del colegio: oficial o no oficial.",
    "cole_area_ubicacion": "Área de ubicación del colegio: urbana o rural.",
    "cole_jornada": "Jornada académica del colegio.",
    "fami_estratovivienda": "Estrato socioeconómico de la vivienda del estudiante.",
    "fami_educacionmadre": "Nivel educativo reportado de la madre.",
    "fami_tieneinternet": "Disponibilidad de internet en el hogar del estudiante.",
    "punt_global": "Puntaje global obtenido en Saber 11, entre 0 y 500.",
    "punt_matematicas": "Puntaje de Matemáticas, entre 0 y 100.",
    "punt_lectura_critica": "Puntaje de Lectura Crítica, entre 0 y 100.",
    "punt_c_naturales": "Puntaje de Ciencias Naturales, entre 0 y 100.",
    "punt_sociales_ciudadanas": "Puntaje de Sociales y Ciudadanas, entre 0 y 100.",
    "punt_ingles": "Puntaje de Inglés, entre 0 y 100.",
    "puntajes_todos_cero": "Indicador de registros con todos los puntajes iguales a cero."
}

# Cargar el diccionario generado anteriormente
ruta_diccionario = "data/oro/diccionario_datos.csv"

diccionario = pd.read_csv(ruta_diccionario)

# Incorporar las descripciones
diccionario["descripcion"] = (
    diccionario["columna"].map(descripciones).fillna("")
)

# Verificar si existen variables sin descripción
sin_descripcion = diccionario[
    diccionario["descripcion"] == ""
]["columna"].tolist()

print("Columnas del diccionario:", len(diccionario))
print("Columnas sin descripción:", sin_descripcion)

# Guardar el diccionario actualizado
diccionario.to_csv(
    ruta_diccionario,
    index=False,
    encoding="utf-8-sig"
)

display(diccionario)

print("Diccionario de datos actualizado correctamente.")

Columnas del diccionario: 19
Columnas sin descripción: []


,columna,tipo,nulos,porcentaje_nulos,descripcion
0,periodo,int,0,0.0,Periodo de aplicación de la prueba Saber 11.
1,anio,int,0,0.0,"Año de aplicación, derivado del periodo."
2,ciclo,int,0,0.0,"Ciclo de aplicación, derivado del periodo."
3,cole_cod_depto_ubicacion,int,0,0.0,Código DANE del departamento donde se ubica el...
4,cole_cod_mcpio_ubicacion,int,0,0.0,Código DANE del municipio donde se ubica el co...
5,cole_mcpio_ubicacion,string,0,0.0,Nombre del municipio donde se ubica el colegio.
6,cole_naturaleza,string,0,0.0,Naturaleza del colegio: oficial o no oficial.
7,cole_area_ubicacion,string,0,0.0,Área de ubicación del colegio: urbana o rural.
8,cole_jornada,string,0,0.0,Jornada académica del colegio.
9,fami_estratovivienda,string,0,0.0,Estrato socioeconómico de la vivienda del estu...


Diccionario de datos actualizado correctamente.
